In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

print("Loading")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")


from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from cols_to_keep import *

from analysis_village.cc1pi.TLExtensionMethod.GaussianFactorFittingUtils import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
#Load CV dataframe
keys2load = ["pfp", "hdr", "histpotdf","hit0","hit1","hit2"] ## keys from the configuration file
#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_stopping.df"
bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_muon_update_calo.df"
mc_bnb_df = load_df(bnb_path, keys2load, 4)
mc_bnb_pfp_df = mc_bnb_df['pfp']
mc_bnb_hit0_df = mc_bnb_df['hit0']
mc_bnb_hit1_df = mc_bnb_df['hit1']
mc_bnb_hit2_df = mc_bnb_df['hit2']
mc_bnb_hdr_df = mc_bnb_df['hdr']

#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_data.df"
bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_5e18_muon_data_update_calo.df"
data_df = load_df(bnb_path, keys2load, 100)
data_pfp_df = data_df['pfp']
data_hit0_df = data_df['hit0']
data_hit1_df = data_df['hit1']
data_hit2_df = data_df['hit2']
data_hdr_df = data_df['hdr']

PDG = 13

In [ ]:
hfit = load_physics_classes()

In [ ]:

conv_tf1_map                    = hfit.get_conv_function_map(PDG, "og", 500)
conv_shifted_tf1_map            = hfit.get_conv_function_map(PDG, "shift", 500)

conv_tf1_map_data                    = hfit.get_conv_function_map(PDG, "og", 500, True)
conv_shifted_tf1_map_data            = hfit.get_conv_function_map(PDG, "shift", 500, True)


In [ ]:
def get_conv_tf1_for_slice(conv_map, plane, rr_center, max_rr):
    """Look up the TF1* in a conv_tf1_map / conv_shifted_tf1_map for the
    residual-range bin closest to rr_center. Bins are 1 cm wide, centered
    at i_rr + 0.5 for i_rr = 0 .. max_rr-1 (see get_conv_function_map).
    Returns None if plane is missing or the index falls outside the map.
    """
    if plane not in conv_map:
        return None
    i_rr = int(np.floor(rr_center))   # rr_center == i_rr + 0.5 for an aligned bin
    if i_rr < 0 or i_rr >= max_rr or i_rr >= len(conv_map[plane]):
        return None
    return conv_map[plane][i_rr]

In [ ]:
def plot_slice_diagnostic_mc_vs_data(
    mc_df,
    data_df,
    plane,
    tpc,
    rr_min,
    rr_max,
    conv_tf1_map=None,             # MC, unshifted
    conv_shifted_tf1_map=None,     # MC, shifted
    conv_tf1_map_data=None,        # data, unshifted
    conv_shifted_tf1_map_data=None,# data, shifted
    dedx_col="dedx",
    nbins=100,
    hist_xmin=0.0,
    hist_xmax=20.0,
    conv_max_rr=500,
    x_limits=None,
    target_height=1.0,
):
    """Overlay MC (blue) and data (red) dE/dx slice histograms for the same
    rr window, both rescaled to share the same peak height (target_height),
    each with its unshifted (dashed) and shifted (solid) convolution TF1
    curves drawn in the matching color and normalized to that same height."""

    def get_slice(df):
        sl = df if tpc == -1 else df[df["tpc"] == tpc]
        return sl[(sl["rr"] >= rr_min) & (sl["rr"] < rr_max) & (sl["pitch"] <= 2)]

    sl_mc = get_slice(mc_df)
    sl_data = get_slice(data_df)

    rr_center = 0.5 * (rr_min + rr_max)
    i_rr_lookup = int(np.floor(rr_center))
    i_rr_lookup = max(0, min(i_rr_lookup, conv_max_rr - 1))
    rr_aligned = i_rr_lookup + 0.5

    fig, ax = plt.subplots(figsize=(8, 5.5))
    bin_edges = np.linspace(hist_xmin, hist_xmax, nbins + 1)
    bin_width = (hist_xmax - hist_xmin) / nbins
    x_eval = np.linspace(hist_xmin, hist_xmax, 400)

    mc_vals = sl_mc[dedx_col].dropna().values
    data_vals = sl_data[dedx_col].dropna().values

    # --- Precompute raw counts to find each histogram's own peak ---
    raw_counts_mc, _ = np.histogram(mc_vals, bins=bin_edges)
    raw_counts_data, _ = np.histogram(data_vals, bins=bin_edges)
    raw_peak_mc = raw_counts_mc.max() if len(raw_counts_mc) else 0.0
    raw_peak_data = raw_counts_data.max() if len(raw_counts_data) else 0.0

    # --- Per-entry weights so each histogram's peak lands at target_height ---
    w_mc = np.full(len(mc_vals), target_height / raw_peak_mc) if raw_peak_mc > 0 else np.ones(len(mc_vals))
    w_data = np.full(len(data_vals), target_height / raw_peak_data) if raw_peak_data > 0 else np.ones(len(data_vals))

    # --- Histograms (now both peak at target_height) ---
    counts_mc, edges, _ = ax.hist(
        mc_vals,
        bins=bin_edges,
        weights=w_mc,
        histtype="stepfilled",
        alpha=0.25,
        color="steelblue",
        edgecolor="navy",
        label=f"MC (N={len(sl_mc)})",
    )
    counts_data, _, _ = ax.hist(
        data_vals,
        bins=bin_edges,
        weights=w_data,
        histtype="stepfilled",
        alpha=0.25,
        color="salmon",
        edgecolor="darkred",
        label=f"Data (N={len(sl_data)})",
    )

    curve_max = target_height
    all_y_curves = []  # for zoom rescaling later

    def draw_conv_curve(tf1_map, color, linestyle, label_prefix):
        if tf1_map is None:
            return None
        f_conv = get_conv_tf1_for_slice(tf1_map, plane, rr_center, conv_max_rr)
        if f_conv is None:
            return None
        y = np.array([f_conv.Eval(xv) for xv in x_eval])
        if y.max() > 0:
            y = y / y.max() * target_height
        mpv = f_conv.GetMaximumX(0.0, 20.0)
        ax.plot(
            x_eval, y,
            color=color, linestyle=linestyle, lw=2.0,
            label=f"{label_prefix} (MPV={mpv:.2f})",
        )
        all_y_curves.append(y)
        return y

    # MC: unshifted dashed, shifted solid, blue
    draw_conv_curve(conv_tf1_map, "steelblue", "--", "MC conv (unshifted)")
    draw_conv_curve(conv_shifted_tf1_map, "steelblue", "-", "MC conv (shifted)")

    # Data: unshifted dashed, shifted solid, red
    draw_conv_curve(conv_tf1_map_data, "crimson", "--", "Data conv (unshifted)")
    draw_conv_curve(conv_shifted_tf1_map_data, "crimson", "-", "Data conv (shifted)")

    curve_max = max([curve_max] + [y.max() for y in all_y_curves]) if all_y_curves else curve_max

    # --- Zoom limits & dynamic y-rescaling ---
    bin_centers = 0.5 * (edges[:-1] + edges[1:])
    if x_limits is not None:
        ax.set_xlim(x_limits)
        mask = (x_eval >= x_limits[0]) & (x_eval <= x_limits[1])
        hist_mask = (bin_centers >= x_limits[0]) & (bin_centers <= x_limits[1])

        visible_max = 0.0
        if len(counts_mc) and np.any(hist_mask):
            visible_max = max(visible_max, counts_mc[hist_mask].max())
        if len(counts_data) and np.any(hist_mask):
            visible_max = max(visible_max, counts_data[hist_mask].max())
        for y_arr in all_y_curves:
            if len(y_arr[mask]) > 0:
                visible_max = max(visible_max, y_arr[mask].max())

        ax.set_ylim(0, visible_max * 1.15 if visible_max > 0 else 1.0)
    else:
        ax.set_ylim(0, curve_max * 1.15 if curve_max > 0 else 1.0)

    ax.set_xlabel(r"$dE/dx$ [MeV/cm]")
    ax.set_ylabel("normalized (peak = 1)")
    tpc_str = "TPCs Combined" if tpc == -1 else f"tpc {tpc}"
    ax.set_title(f"plane {plane}, {tpc_str}, {rr_min:g} <= rr < {rr_max:g} cm")
    ax.legend(fontsize=9)
    fig.tight_layout()
    plt.show()
    return fig

In [ ]:
mc_hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]
data_hit_dfs = [data_hit0_df, data_hit1_df, data_hit2_df]
rr_ranges = [(1, 2), (2, 3), (3, 4), (4, 5), (6, 7), (15, 16), (30, 31)]

plane = 2
mc_df = mc_hit_dfs[plane]
data_df_slice = data_hit_dfs[plane]

for rr_min, rr_max in rr_ranges:
    plot_slice_diagnostic_mc_vs_data(
        mc_df, data_df_slice,
        plane=plane, tpc=-1, rr_min=rr_min, rr_max=rr_max,
        conv_tf1_map=conv_tf1_map,
        conv_shifted_tf1_map=conv_shifted_tf1_map,
        conv_tf1_map_data=conv_tf1_map_data,
        conv_shifted_tf1_map_data=conv_shifted_tf1_map_data,
        conv_max_rr=500,
        x_limits=(1.5, 4),
    )